# Reproduction notebook: 40_full_long_horizon_relevance_to_downstream_bridge_v2

This notebook is retained as an executable provenance record for the anonymous supplementary package. Saved outputs and internal development notes have been removed.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 240)

DATASETS = [
    "ETTh1",
    "Weather",
    "Electricity",
    "Traffic",
    "Exchange",
    "Solar",
]

HORIZONS = [96, 192, 336, 720]

BACKBONES = [
    "PatchTST",
    "iTransformer",
    "TimeMixer",
    "SegMoE",
]

ROOT_CANDIDATES = [
    Path("/data/dataset/strong_forecaster"),
    Path("/data/strong_forecaster"),
]

ROOT = next((p for p in ROOT_CANDIDATES if p.exists()), None)

if ROOT is None:
    raise FileNotFoundError(
        "Could not find strong_forecaster root. Tried:\n"
        + "\n".join(f" - {p}" for p in ROOT_CANDIDATES)
    )

OUT_DIR = ROOT / "full_long_horizon_relevance_downstream_bridge"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)


In [ ]:
rows = [
    # Dataset, H, Pattern, Prior, Learned, Shuffled, Sig Pattern->Learned, Sig Shuffled->Learned
    ('ETTh1', 96, 1.195878, 1.2183, 0.995375, 1.387354, True, True),
    ('ETTh1', 192, 1.383055, 1.292313, 1.160534, 1.537791, True, True),
    ('ETTh1', 336, 1.556203, 1.356678, 1.332749, 1.610606, True, True),
    ('ETTh1', 720, 1.737053, 1.490594, 1.532154, 1.78954, True, True),

    ('Weather', 96, 0.913355, 0.25785, 0.806514, 0.627484, True, False),
    ('Weather', 192, 0.962618, 0.320045, 0.877885, 0.786918, True, False),
    ('Weather', 336, 1.100516, 0.406713, 0.980122, 0.968104, True, False),
    ('Weather', 720, 1.273764, 0.524525, 1.157938, 1.140886, True, False),

    ('Electricity', 96, 0.477137, 0.41176, 0.392364, 0.554495, True, True),
    ('Electricity', 192, 0.482181, 0.417895, 0.408721, 0.550554, True, True),
    ('Electricity', 336, 0.513699, 0.440371, 0.442422, 0.57982, True, True),
    ('Electricity', 720, 0.58904, 0.50018, 0.519661, 0.667902, True, True),

    ('Traffic', 96, 0.875554, 0.905708, 0.863432, 1.092021, False, True),
    ('Traffic', 192, 0.883992, 0.914254, 0.869169, 1.084658, True, True),
    ('Traffic', 336, 0.910878, 0.940618, 0.894336, 1.102242, True, True),
    ('Traffic', 720, 0.97161, 0.998918, 0.948603, 1.137833, True, True),

    ('Exchange', 96, 0.211965, 0.098876, 0.201246, 0.199855, False, False),
    ('Exchange', 192, 0.424098, 0.228619, 0.414987, 0.423465, False, True),
    ('Exchange', 336, 0.769549, 0.469822, 0.823094, 0.79739, False, False),
    ('Exchange', 720, 1.894643, 1.514171, 2.06649, 1.935582, False, False),

    ('Solar', 96, 0.666227, 1.016373, 0.471276, 1.316049, True, True),
    ('Solar', 192, 0.64461, 0.990156, 0.498956, 1.434488, True, True),
    ('Solar', 336, 0.693457, 1.072159, 0.526908, 1.547378, True, True),
    ('Solar', 720, 0.726212, 1.165314, 0.56462, 1.607708, True, True),
]

rel = pd.DataFrame(
    rows,
    columns=[
        "Dataset",
        "Horizon",
        "Pattern",
        "CandidatePrior",
        "Learned",
        "Shuffled",
        "Sig_RetrievalGain",
        "Sig_QuerySpecificGain",
    ],
)

rel["RetrievalGain_pct"] = (
    (rel["Pattern"] - rel["Learned"])
    / rel["Pattern"]
    * 100.0
)

rel["QuerySpecificGain_pct"] = (
    (rel["Shuffled"] - rel["Learned"])
    / rel["Shuffled"]
    * 100.0
)

rel["PriorGain_pct"] = (
    (rel["Pattern"] - rel["CandidatePrior"])
    / rel["Pattern"]
    * 100.0
)

display(rel)

print(
    "Learned beats Pattern:",
    int((rel["RetrievalGain_pct"] > 0).sum()),
    "/",
    len(rel),
)

print(
    "Significant Pattern->Learned:",
    int(rel["Sig_RetrievalGain"].sum()),
    "/",
    len(rel),
)


In [ ]:
preferred = (
    ROOT
    / "four_backbone_dataset_meta_analysis"
    / "condition_level_selected.csv"
)

if preferred.is_file():
    condition_csv = preferred
else:
    hits = sorted(ROOT.rglob("condition_level_selected.csv"))

    if not hits:
        raise FileNotFoundError(
            "Could not find condition_level_selected.csv. "
            "Run Experiment 37 v3 first."
        )

    hits = sorted(
        hits,
        key=lambda p: (
            0 if "four_backbone_dataset_meta_analysis" in str(p) else 1,
            len(p.parts),
            str(p),
        ),
    )

    condition_csv = hits[0]

print("Using:", condition_csv)

conditions = pd.read_csv(condition_csv)

required = {
    "Dataset",
    "Backbone",
    "Horizon",
    "MSEGain_pct",
}

missing = required - set(conditions.columns)

if missing:
    raise ValueError(
        f"Missing required columns: {sorted(missing)}"
    )

down = conditions[
    conditions["Dataset"].isin(DATASETS)
    & conditions["Backbone"].isin(BACKBONES)
    & conditions["Horizon"].astype(int).isin(HORIZONS)
].copy()

coverage = (
    down.groupby(["Dataset", "Horizon"])
    ["Backbone"]
    .nunique()
    .reindex(
        pd.MultiIndex.from_product(
            [DATASETS, HORIZONS],
            names=["Dataset", "Horizon"],
        )
    )
)

display(coverage.rename("N_Backbones").reset_index())

if not bool((coverage == 4).all()):
    raise RuntimeError(
        "Downstream coverage is incomplete.\n"
        + str(coverage)
    )

print("PASS: 24 dataset-horizon cells × 4 backbones = 96 downstream conditions.")


In [ ]:
down_rows = []

for dataset in DATASETS:
    for H in HORIZONS:
        grp = down[
            (down["Dataset"] == dataset)
            & (down["Horizon"].astype(int) == H)
        ].copy()

        row = {
            "Dataset": dataset,
            "Horizon": H,
            "DownstreamMeanGain_pct": float(grp["MSEGain_pct"].mean()),
            "DownstreamMedianGain_pct": float(grp["MSEGain_pct"].median()),
            "DownstreamWins": int((grp["MSEGain_pct"] > 0).sum()),
            "DownstreamLosses": int((grp["MSEGain_pct"] < 0).sum()),
            "DownstreamTies": int((grp["MSEGain_pct"].abs() <= 1e-12).sum()),
        }

        for backbone in BACKBONES:
            bb = grp[grp["Backbone"] == backbone]

            if len(bb) != 1:
                raise RuntimeError(
                    f"Expected one row for {dataset}/{H}/{backbone}; found {len(bb)}."
                )

            row[f"{backbone}_Gain_pct"] = float(
                bb.iloc[0]["MSEGain_pct"]
            )

        down_rows.append(row)

down_h = pd.DataFrame(down_rows)

bridge = rel.merge(
    down_h,
    on=["Dataset", "Horizon"],
    how="inner",
    validate="one_to_one",
)

display(
    bridge[
        [
            "Dataset",
            "Horizon",
            "RetrievalGain_pct",
            "QuerySpecificGain_pct",
            "PriorGain_pct",
            "DownstreamMeanGain_pct",
            "DownstreamWins",
            "DownstreamTies",
            "DownstreamLosses",
        ]
    ]
)

bridge.to_csv(
    OUT_DIR / "full_24point_bridge.csv",
    index=False,
)


In [ ]:
def corr_summary(df, label):
    return {
        "Level": label,
        "N": len(df),
        "Pearson_Retrieval_vs_Downstream": float(
            df["RetrievalGain_pct"].corr(
                df["DownstreamMeanGain_pct"],
                method="pearson",
            )
        ),
        "Spearman_Retrieval_vs_Downstream": float(
            df["RetrievalGain_pct"].corr(
                df["DownstreamMeanGain_pct"],
                method="spearman",
            )
        ),
        "Pearson_QuerySpecific_vs_Downstream": float(
            df["QuerySpecificGain_pct"].corr(
                df["DownstreamMeanGain_pct"],
                method="pearson",
            )
        ),
        "Spearman_QuerySpecific_vs_Downstream": float(
            df["QuerySpecificGain_pct"].corr(
                df["DownstreamMeanGain_pct"],
                method="spearman",
            )
        ),
    }


dataset_mean = (
    bridge.groupby("Dataset", as_index=False)
    .agg(
        RetrievalGain_pct=("RetrievalGain_pct", "mean"),
        QuerySpecificGain_pct=("QuerySpecificGain_pct", "mean"),
        PriorGain_pct=("PriorGain_pct", "mean"),
        DownstreamMeanGain_pct=("DownstreamMeanGain_pct", "mean"),
        DownstreamWins=("DownstreamWins", "sum"),
        DownstreamLosses=("DownstreamLosses", "sum"),
        DownstreamTies=("DownstreamTies", "sum"),
    )
)

corr = pd.DataFrame([
    corr_summary(bridge, "24 dataset-horizon cells"),
    corr_summary(dataset_mean, "6 dataset means"),
])

display(corr)
display(dataset_mean)

corr.to_csv(
    OUT_DIR / "bridge_correlations_descriptive.csv",
    index=False,
)

dataset_mean.to_csv(
    OUT_DIR / "bridge_dataset_means.csv",
    index=False,
)


In [ ]:
bridge["RetrievalImproves"] = bridge["RetrievalGain_pct"] > 0
bridge["DownstreamImproves"] = bridge["DownstreamMeanGain_pct"] > 0

counter = bridge[
    bridge["RetrievalImproves"]
    & (~bridge["DownstreamImproves"])
].copy()

display(
    counter[
        [
            "Dataset",
            "Horizon",
            "RetrievalGain_pct",
            "DownstreamMeanGain_pct",
            "DownstreamWins",
            "DownstreamLosses",
            "QuerySpecificGain_pct",
            "PriorGain_pct",
        ]
    ]
)

print(
    "Positive relevance but non-positive mean downstream utility:",
    len(counter),
    "/",
    int(bridge["RetrievalImproves"].sum()),
)

for dataset in DATASETS:
    g = bridge[bridge["Dataset"] == dataset]

    print(
        f"{dataset:12s} | "
        f"relevance positive {int((g['RetrievalGain_pct'] > 0).sum())}/4 | "
        f"downstream mean positive {int((g['DownstreamMeanGain_pct'] > 0).sum())}/4 | "
        f"backbone wins {int(g['DownstreamWins'].sum())}/16"
    )


In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 5.4))

for dataset in DATASETS:
    g = bridge[
        bridge["Dataset"] == dataset
    ].sort_values("Horizon")

    ax.plot(
        g["RetrievalGain_pct"],
        g["DownstreamMeanGain_pct"],
        marker="o",
        label=dataset,
    )

    for _, r in g.iterrows():
        ax.annotate(
            str(int(r["Horizon"])),
            (
                r["RetrievalGain_pct"],
                r["DownstreamMeanGain_pct"],
            ),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
        )

ax.axhline(0.0, linewidth=1.0)
ax.axvline(0.0, linewidth=1.0)

ax.set_xlabel(
    "Predictive-relevance improvement: Pattern → Learned (%)"
)

ax.set_ylabel(
    "Downstream MSE improvement across 4 backbones (%)"
)

ax.set_title(
    "Same-horizon bridge: relevance and downstream utility are distinct"
)

ax.legend(
    ncol=2,
    fontsize=8,
)

fig.tight_layout()

fig.savefig(
    OUT_DIR / "full_24point_relevance_downstream_bridge.pdf",
    bbox_inches="tight",
)

fig.savefig(
    OUT_DIR / "full_24point_relevance_downstream_bridge.png",
    dpi=240,
    bbox_inches="tight",
)

plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 5.4))

for dataset in DATASETS:
    g = bridge[
        bridge["Dataset"] == dataset
    ].sort_values("Horizon")

    ax.plot(
        g["QuerySpecificGain_pct"],
        g["DownstreamMeanGain_pct"],
        marker="o",
        label=dataset,
    )

    for _, r in g.iterrows():
        ax.annotate(
            str(int(r["Horizon"])),
            (
                r["QuerySpecificGain_pct"],
                r["DownstreamMeanGain_pct"],
            ),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
        )

ax.axhline(0.0, linewidth=1.0)
ax.axvline(0.0, linewidth=1.0)

ax.set_xlabel(
    "Query-specific evidence: Shuffled → Correct Learned (%)"
)

ax.set_ylabel(
    "Downstream MSE improvement across 4 backbones (%)"
)

ax.set_title(
    "Query-specific relevance is not sufficient for downstream complementarity"
)

ax.legend(
    ncol=2,
    fontsize=8,
)

fig.tight_layout()

fig.savefig(
    OUT_DIR / "query_specific_evidence_downstream_bridge.pdf",
    bbox_inches="tight",
)

fig.savefig(
    OUT_DIR / "query_specific_evidence_downstream_bridge.png",
    dpi=240,
    bbox_inches="tight",
)

plt.show()


In [ ]:
table24 = bridge[
    [
        "Dataset",
        "Horizon",
        "RetrievalGain_pct",
        "QuerySpecificGain_pct",
        "PriorGain_pct",
        "DownstreamMeanGain_pct",
        "DownstreamWins",
    ]
].copy()

for col in [
    "RetrievalGain_pct",
    "QuerySpecificGain_pct",
    "PriorGain_pct",
    "DownstreamMeanGain_pct",
]:
    table24[col] = table24[col].map(
        lambda v: f"{v:+.2f}"
    )

table24["DownstreamWins"] = table24["DownstreamWins"].map(
    lambda v: f"{int(v)}/4"
)

display(table24)

latex24 = table24.to_latex(
    index=False,
    escape=False,
)

(OUT_DIR / "full_24point_bridge_table.tex").write_text(
    latex24,
    encoding="utf-8",
)

table6 = dataset_mean[
    [
        "Dataset",
        "RetrievalGain_pct",
        "QuerySpecificGain_pct",
        "PriorGain_pct",
        "DownstreamMeanGain_pct",
        "DownstreamWins",
        "DownstreamTies",
        "DownstreamLosses",
    ]
].copy()

for col in [
    "RetrievalGain_pct",
    "QuerySpecificGain_pct",
    "PriorGain_pct",
    "DownstreamMeanGain_pct",
]:
    table6[col] = table6[col].map(
        lambda v: f"{v:+.2f}"
    )

display(table6)

latex6 = table6.to_latex(
    index=False,
    escape=False,
)

(OUT_DIR / "dataset_mean_bridge_table.tex").write_text(
    latex6,
    encoding="utf-8",
)

print("Saved LaTeX tables to:", OUT_DIR)


In [ ]:
print("=" * 116)
print("EXPERIMENT 40 — FULL LONG-HORIZON RELEVANCE → DOWNSTREAM BRIDGE")
print("=" * 116)

print(
    "Retrieval gain > 0:",
    int((bridge["RetrievalGain_pct"] > 0).sum()),
    "/24",
)

print(
    "Significant Pattern->Learned relevance gain:",
    int(bridge["Sig_RetrievalGain"].sum()),
    "/24",
)

print(
    "Significant Correct-vs-Shuffled evidence:",
    int(bridge["Sig_QuerySpecificGain"].sum()),
    "/24",
)

print()

for _, r in dataset_mean.iterrows():
    print(
        f"{r['Dataset']:12s} | "
        f"retrieval={r['RetrievalGain_pct']:+7.2f}% | "
        f"query-specific={r['QuerySpecificGain_pct']:+7.2f}% | "
        f"prior={r['PriorGain_pct']:+7.2f}% | "
        f"downstream={r['DownstreamMeanGain_pct']:+7.3f}% | "
        f"wins={int(r['DownstreamWins'])}/16"
    )

print("\nOutputs:")
for name in [
    "full_24point_bridge.csv",
    "bridge_correlations_descriptive.csv",
    "bridge_dataset_means.csv",
    "full_24point_relevance_downstream_bridge.pdf",
    "full_24point_relevance_downstream_bridge.png",
    "query_specific_evidence_downstream_bridge.pdf",
    "query_specific_evidence_downstream_bridge.png",
    "full_24point_bridge_table.tex",
    "dataset_mean_bridge_table.tex",
]:
    print(" -", OUT_DIR / name)
